# LoRA Finetune Studio - Quick Start

This notebook walks you through the complete workflow for fine-tuning an LLM using LoRA.

## Steps
1. Install dependencies
2. Prepare your dataset
3. Configure training
4. Run fine-tuning
5. Generate text with the fine-tuned model

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate gradio pandas

In [ ]:
import sys
sys.path.insert(0, '../src')

from lorakit.config import FullConfig, ModelConfig, LoRAConfig, DataConfig, TrainingConfig
from lorakit.data.dataset import DatasetLoader
from lorakit.data.preprocessor import TextPreprocessor
from lorakit.training.trainer import LoRATrainer
from lorakit.inference.generator import TextGenerator

print('Imports successful')

## Configuration

Customize the configuration for your use case.

In [ ]:
config = FullConfig(
    model=ModelConfig(
        model_name='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        use_4bit=True,
    ),
    lora=LoRAConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0.05,
    ),
    data=DataConfig(
        train_file='../train.csv',
        prompt_column='instruction',
        response_column='output',
        max_seq_length=512,
        val_split=0.1,
    ),
    training=TrainingConfig(
        output_dir='../outputs/quickstart',
        num_train_epochs=1,
        per_device_train_batch_size=2,
        learning_rate=2e-4,
    ),
)
print('Config created:', config)

## Load and Inspect Dataset

In [ ]:
loader = DatasetLoader(config.data)
dataset = loader.load(file_path=config.data.train_file)
stats = loader.get_dataset_stats(dataset)
print('Dataset stats:', stats)
print('Sample:', dataset['train'][0])

## Preview Formatted Prompt

In [ ]:
preprocessor = TextPreprocessor(config.data)
formatter = preprocessor.get_formatter('instruction')
sample = dataset['train'][0]
formatted = formatter(sample)
print(formatted['text'])

## Train the Model

This will download the model and start training. Requires a GPU.

In [ ]:
trainer = LoRATrainer(config)

def on_log(entry):
    print(f"Step {entry.get('step')}: loss={entry.get('loss', 'N/A')}")

result = trainer.train(dataset, prompt_template='instruction', on_log=on_log)
print('Training result:', result)

## Generate Text with the Fine-tuned Model

In [ ]:
generator = TextGenerator(
    model_path='../outputs/quickstart',
    base_model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    load_in_4bit=True,
)
generator.load()

prompt = 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a Midjourney prompt for a futuristic city\n\n### Response:\n'
outputs = generator.generate(prompt, max_new_tokens=200, temperature=0.7)
print(outputs[0])